In [ ]:
%load_ext autoreload
%autoreload 2

PG_WRITER_CONFIG = "../configs/writer/postgres.json"


# Completeness Metric

In [ ]:
from metis.metric.completeness.completeness_nullRate import completeness_nullRate
from metis.metric.completeness.completeness_nullAndDMVRate_config import (
    completeness_nullAndDMVRate_config,
)
from metis.metric.completeness.completeness_nullAndDMVRate import (
    completeness_nullAndDMVRate,
)
from metis.metric.completeness.completeness_nullRate_config import (
    completeness_nullRate_config,
)
from utils import execute_run

execute_run(
    folder="/Users/jberndt/Documents/Masterarbeit/Metis/data/local/polluted/20260105_122849/completeness",
    metrics=[completeness_nullAndDMVRate.__name__, completeness_nullRate.__name__],
    metric_configs=[
        completeness_nullAndDMVRate_config(
            aggregation_axis="columns", aggregate_all=False
        ),
        completeness_nullRate_config(aggregation_axis="columns", aggregate_all=False),
    ],
)

INFO:metis:Assessing metric 'completeness_nullAndDMVRate'
/var/folders/rw/xym_kq6n3l9_0qkvb5l3bd_w0000gn/T/tmp7fm__ace.csv  has :99516  tuples  and  23  attributes.


# Rule Consistency Metric

In [ ]:
import re
from typing import List

import pandas as pd
from metis.metric.config import MetricConfig
from metis.metric.consistency.consistency_ruleBasedPipino_config import (
    consistency_ruleBasedPipino_config,
)
from metis.metric.consistency.consistency_ruleBasedPipino import (
    consistency_ruleBasedPipino,
)
from metis.utils.datetime.datetime_precision import determine_datetime_precision
from utils import execute_run

NUMBER_REGEX = re.compile(r"^\-?\d+(\.\d+)?")


def extract_number(value: str):
    match = NUMBER_REGEX.match(value)
    if match:
        return float(match.group(0))
    raise ValueError(f"Could not extract number from value: {value}")


def is_number(value: str) -> bool:
    try:
        float(value)
        return True
    except ValueError:
        return False


def is_datetime(value: str) -> bool:
    try:
        pd.to_datetime(value)
        return True
    except (ValueError, TypeError):
        return False


def contains_expected_datetime_format(value: str, format: str) -> bool:
    try:
        pd.to_datetime(value, exact=False, format=format)
        return True
    except (ValueError, TypeError):
        return False


temp_rules = [
    lambda value: (extract_number(value) >= -10 and extract_number(value) <= 50),
    lambda value: str(extract_number(value))[::-1].find(".") == 1,
    lambda value: is_number(value),
]


def is_duration_format(value: str) -> bool:
    parts = value.split(" ")
    if len(parts) != 2:
        return False
    number_part = parts[0]
    if not is_number(number_part):
        return False
    return True


def is_minute_unit(value: str) -> bool:
    parts = value.split(" ")
    if len(parts) != 2:
        return False
    unit_part = parts[1]
    return unit_part in ["min", "m"]


def is_min_abbr(value: str) -> bool:
    parts = value.split(" ")
    if len(parts) != 2:
        return False
    unit_part = parts[1]
    return unit_part == "min"


def get_datetime_part(value: str) -> str:
    location = re.search(r"\(.*\)", value)
    if not location:
        return value
    return value.replace(location.group(0), "").strip()


def is_datetime_with_location(value: str) -> bool:
    location = re.search(r"\(.*\)", value)
    if not location:
        return False
    dt_part = get_datetime_part(value)
    return is_datetime(dt_part)


def location_is_at_end(value: str) -> bool:
    match = re.search(r"\(.*\)", value)
    if not match:
        return False
    return match.end() == len(value)


metrics = [consistency_ruleBasedPipino.__name__]
metric_configs: List[str | None | MetricConfig] = [
    consistency_ruleBasedPipino_config(
        attribute_rules={
            "MinTemp": temp_rules,
            "MaxTemp": temp_rules,
            "PRICEEACH": [
                lambda value: is_number(value),
                lambda value: is_number(value)
                or (not is_number(value) and "$" in value),
            ],
            "ORDERDATE": [
                lambda value: is_datetime(value),
                lambda value: (
                    is_datetime(value)
                    and contains_expected_datetime_format(value, "%d/%m/%Y")
                ),
                lambda value: (
                    is_datetime(value) and determine_datetime_precision(value) == "day"
                ),
            ],
            "Duration": [
                lambda value: is_duration_format(value),
                lambda value: is_minute_unit(value),
                lambda value: is_min_abbr(value),
            ],
            "Release Date": [
                lambda value: is_datetime_with_location(value),
                lambda value: location_is_at_end(value),
                lambda value: contains_expected_datetime_format(
                    get_datetime_part(value), "DD MMMM YYYY"
                ),
                lambda value: determine_datetime_precision(get_datetime_part(value))
                == "day",
            ],
        },
        # tuple_rules=[
        #     lambda row: 1.0 if (
        #         pd.isna(row["MinTemp"]) or pd.isna(row["MaxTemp"])
        #     ) else (0.0 if row["MinTemp"] <= row["MaxTemp"] else 1.0)
        # ]
    ),
]

execute_run(
    folder="/Users/jberndt/Documents/Masterarbeit/Metis/data/local/polluted/20260107_180606/consistency",
    metrics=metrics,
    metric_configs=metric_configs,
)

INFO:metis:Assessing metric 'consistency_ruleBasedPipino'


KeyboardInterrupt: 

# Currency

In [5]:
from typing import List

from metis.metric.config import MetricConfig
from metis.metric.timeliness.timeliness_heinrich_config import (
    timeliness_heinrich_config,
)
from metis.metric.timeliness.timeliness_heinrich import timeliness_heinrich
from notebooks.utils import execute_run

metrics = [timeliness_heinrich.__name__]
metric_configs: List[str | None | MetricConfig] = [
    timeliness_heinrich_config(
        decline_rate_per_column={
            "ADDRESSLINE1": 0.1 / 365.25,
        },
        ingestion_date_column="ORDERDATE",
        simulated_assessment_date="2021-05-30",  # newest entry in auto sales data: 2020-05-30T22:00:00.000Z
    ),
]

execute_run(
    folder="/Users/jberndt/Documents/Masterarbeit/Metis/data/local/polluted/20260107_180606/timeliness",
    metrics=metrics,
    metric_configs=metric_configs,
)

INFO:metis:Assessing metric 'timeliness_heinrich'
/Users/jberndt/Documents/Masterarbeit/Metis/notebooks/../metis/metric/timeliness/timeliness_heinrich.py:60: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  ingestion_dates = pd.to_datetime(data[ingestion_date_column])
INFO:metis:Writing 2747 results using CSVWriter
INFO:metis:Results saved to /Users/jberndt/Documents/Masterarbeit/Metis/notebooks/results/20260115_073742/dq_results.csv


DQ run completed in 0:00:00.088915


In [7]:
from typing import List

from metis.metric.config import MetricConfig
from metis.metric.timeliness.timeliness_heinrich_config import (
    timeliness_heinrich_config,
)
from metis.metric.timeliness.timeliness_heinrich import timeliness_heinrich
from notebooks.utils import execute_run

# average total time between revisions (days): 3203.0898
# average total number of revisions: 4.2888
ol_column_change_rates_prev = {
    "author": 1147.09,
    "authors": 2031.67,
    "by_statement": 1070.83,
    "collections": 1147.09,
    "contributions": 1035.68,
    "contributors": 1035.68,
    "copyright_date": 1149.17,
    "coverimage": 1147.42,
    "covers": 1521.72,
    "create": 1147.09,
    "description": 1153.52,
    "dewey_decimal_class": 1150.09,
    "edition_name": 1186.23,
    "first_sentence": 1153.93,
    "full_title": 1248.34,
    "genres": 1139.92,
    "is_box_id": 1158.84,
    "ia_loaded_id": 1154.58,
    "isbn": 1147.11,
    "isbn_10": 1294.91,
    "isbn_13": 1363.01,
    "isbn_invalid": 1145.99,
    "isbn_odd_length": 1147.04,
    "language": 1147.44,
    "languages": 1091.04,
    "lc_classification": 1148.91,
    "lc_classifications": 1406.25,
    "lccn": 1089.36,
    "lexile": 1147.09,
    "links": 1150.49,
    "local_id": 1202.98,
    "location": 1141.77,
    "notes": 1078.43,
    "number_of_pages": 1060.82,
    "ocaid": 1230.22,
    "oclc": 1147.09,
    "oclc_number": 1142.52,
    "oclc_numbers": 1333.89,
    "openlibrary": 1147.09,
    "original_isbn": 1147.23,
    "other_titles": 1132.01,
    "pagination": 1077.22,
    "physical_dimensions": 1175.65,
    "physical_format": 1277.75,
    "providers": 1147.09,
    "publish_country": 883.03,
    "publish_date": 1232.22,
    "publish_places": 869.65,
    "publishers": 2109.29,
    "purchase_url": 1146.91,
    "scan_on_demand": 1146.31,
    "scan_records": 1147.55,
    "series": 1114.83,
    "source_records": 1364.6,
    "subject_people": 1154.08,
    "subject_place": 1097.93,
    "subject_places": 1164.83,
    "subject_time": 1141.76,
    "subject_times": 1153.15,
    "subjects": 1499.95,
    "subtitle": 1136.89,
    "table_of_contents": 1177.92,
    "title": 4617.58,
    "title_prefix": 1284.87,
    "translated_from": 1147.94,
    "translation_of": 1147.37,
    "uri_descriptions": 1177.77,
    "uris": 1177.87,
    "url": 1151.35,
    "volumes": 1147.63,
    "weight": 1207.63,
    "word_count": 1147.09,
    "work_title": 1144.55,
    "work_titles": 1144.97,
    "works": 3329.91,
    "download_url": 1146.61,
    "openstax_id": 1147.23,
    "author_names": 11498.71,
    "bookweight_unit": 1148.73,
    "numer_of_pages": 1148.78,
    "language_code": 1144.36,
}

ol_column_change_rates = {
    "author": {
        "avg_changes": 2.8366738154517856e-05,
        "avg_time": 3238.2092169591638,
        "null_rate": 99.99878974497909,
    },
    "author_names": {
        "avg_changes": 0.0,
        "avg_time": None,
        "null_rate": 99.9999394252547,
    },
    "authors": {
        "avg_changes": 0.08336869484758914,
        "avg_time": 2541.418977296649,
        "null_rate": 12.967036356719841,
    },
    "bookweight_unit": {
        "avg_changes": 1.7397355601948505e-06,
        "avg_time": 5416.584752099259,
        "null_rate": 99.9996365515282,
    },
    "by_statement": {
        "avg_changes": 0.00237272189207989,
        "avg_time": 4138.006353709576,
        "null_rate": 54.89252544274565,
    },
    "by_statements": {"avg_changes": 0.0, "avg_time": None, "null_rate": 100.0},
    "collections": {
        "avg_changes": 5.56881795076628e-06,
        "avg_time": 3563.4601605657053,
        "null_rate": 99.99933443652361,
    },
    "contributions": {
        "avg_changes": 0.0006383296636657999,
        "avg_time": 4269.553478418656,
        "null_rate": 76.92057058864341,
    },
    "contributors": {
        "avg_changes": 0.00312334110458999,
        "avg_time": 1632.672308778373,
        "null_rate": 99.75750717254662,
    },
    "copyright_date": {
        "avg_changes": 0.00303472550135798,
        "avg_time": 1542.3744165388744,
        "null_rate": 99.49114962119211,
    },
    "coverimage": {
        "avg_changes": 5.864378031440131e-05,
        "avg_time": 4408.216621217835,
        "null_rate": 99.98626463189127,
    },
    "covers": {
        "avg_changes": 0.1712877552756112,
        "avg_time": 3900.253777412511,
        "null_rate": 72.12881197606005,
    },
    "create": {"avg_changes": 0.0, "avg_time": None, "null_rate": 99.9982150749281},
    "created": {
        "avg_changes": 0.26271556531560686,
        "avg_time": 4862.114134834187,
        "null_rate": 24.970999978410685,
    },
    "description": {
        "avg_changes": 0.00994849738993504,
        "avg_time": 4075.7180196719555,
        "null_rate": 96.05276996836581,
    },
    "dewey_decimal_class": {
        "avg_changes": 0.00021771336721706,
        "avg_time": 2680.6572119054763,
        "null_rate": 76.66223740383693,
    },
    "download_url": {
        "avg_changes": 3.2627643407150594e-06,
        "avg_time": 3799.9510858689177,
        "null_rate": 99.9990396423993,
    },
    "edition_name": {
        "avg_changes": 0.016353884962166287,
        "avg_time": 2501.5990003755805,
        "null_rate": 84.06724080575512,
    },
    "first_sentence": {
        "avg_changes": 0.00046155171468105,
        "avg_time": 2870.9363006232848,
        "null_rate": 97.66133570909832,
    },
    "full_title": {
        "avg_changes": 0.00011503806475726,
        "avg_time": 3574.5778551611606,
        "null_rate": 83.2900569554944,
    },
    "genres": {
        "avg_changes": 0.0001099850539468249,
        "avg_time": 4225.712424710462,
        "null_rate": 95.42310541778477,
    },
    "ia_box_id": {
        "avg_changes": 0.00875898444275713,
        "avg_time": 4334.2403883045945,
        "null_rate": 98.58047153627967,
    },
    "ia_loaded_id": {
        "avg_changes": 0.00433774291246409,
        "avg_time": 4639.926917446104,
        "null_rate": 99.12732440394582,
    },
    "identfiers_doi": {
        "avg_changes": 0.0,
        "avg_time": None,
        "null_rate": 99.99987910849771,
    },
    "isbn": {
        "avg_changes": 5.864324298359673e-05,
        "avg_time": 4391.287071841471,
        "null_rate": 99.99738594641266,
    },
    "isbn_10": {
        "avg_changes": 0.00159044158387219,
        "avg_time": 2189.290347647592,
        "null_rate": 47.20149176975297,
    },
    "isbn_13": {
        "avg_changes": 0.00259165129364558,
        "avg_time": 2182.517485131629,
        "null_rate": 53.563249936395316,
    },
    "isbn_invalid": {
        "avg_changes": 0.0,
        "avg_time": None,
        "null_rate": 99.82886258027392,
    },
    "isbn_odd_length": {
        "avg_changes": 0.0,
        "avg_time": None,
        "null_rate": 99.99604873428878,
    },
    "language": {
        "avg_changes": 0.00011190499240271001,
        "avg_time": 3964.5964311839402,
        "null_rate": 99.98897503410751,
    },
    "language_code": {
        "avg_changes": 1.7393739485678797e-06,
        "avg_time": 5980.678061930792,
        "null_rate": 99.99951598179824,
    },
    "languages": {
        "avg_changes": 0.01864047809061937,
        "avg_time": 3566.2592093258927,
        "null_rate": 13.743305463482406,
    },
    "last_modified": {
        "avg_changes": 1.8759264397387323,
        "avg_time": 1147.029185088261,
        "null_rate": 0.0,
    },
    "latest_revision": {
        "avg_changes": 1.86341181975586,
        "avg_time": 1151.4106038215334,
        "null_rate": 14.434892714967434,
    },
    "lc_classification": {
        "avg_changes": 0.0,
        "avg_time": None,
        "null_rate": 99.99960669633417,
    },
    "lc_classifications": {
        "avg_changes": 0.10835713698067721,
        "avg_time": 3727.0948526132292,
        "null_rate": 55.0741091424873,
    },
    "lccn": {
        "avg_changes": 0.032324870449509394,
        "avg_time": 4457.0064625789955,
        "null_rate": 63.345670859848234,
    },
    "lexile": {
        "avg_changes": 0.00018394533774025,
        "avg_time": 3804.5748494750223,
        "null_rate": 99.99283542924871,
    },
    "links": {
        "avg_changes": 2.1230009394868718e-05,
        "avg_time": 4344.448730752096,
        "null_rate": 99.1056840266935,
    },
    "local_id": {
        "avg_changes": 0.09899996749834918,
        "avg_time": 3713.7805083007966,
        "null_rate": 90.28531534157707,
    },
    "location": {
        "avg_changes": 2.7147261106778217e-05,
        "avg_time": 4139.313417638165,
        "null_rate": 98.68507551766183,
    },
    "notes": {
        "avg_changes": 0.007310228049437831,
        "avg_time": 4049.597340419559,
        "null_rate": 58.737326911388955,
    },
    "number_of_pages": {
        "avg_changes": 0.02031132170435026,
        "avg_time": 3020.488160175053,
        "null_rate": 28.303092285506636,
    },
    "numer_of_pages": {
        "avg_changes": 0.0,
        "avg_time": None,
        "null_rate": 99.99908970308542,
    },
    "ocaid": {
        "avg_changes": 0.07212246118403358,
        "avg_time": 4536.284327194121,
        "null_rate": 86.09963861283981,
    },
    "oclc": {"avg_changes": 0.0, "avg_time": None, "null_rate": 99.99729498716427},
    "oclc_number": {
        "avg_changes": 1.1314080469074252e-05,
        "avg_time": 5163.568016187406,
        "null_rate": 98.42049875271157,
    },
    "oclc_numbers": {
        "avg_changes": 0.16214507130625946,
        "avg_time": 4415.5663334009805,
        "null_rate": 72.7867346529845,
    },
    "openlibrary": {
        "avg_changes": 1.0618411054909168e-05,
        "avg_time": 1565.1306130988191,
        "null_rate": 99.96893859778633,
    },
    "openstax_id": {
        "avg_changes": 2.1740504315759205e-06,
        "avg_time": 1088.3779651159664,
        "null_rate": 99.99984887581991,
    },
    "original_isbn": {
        "avg_changes": 4.193618230202721e-05,
        "avg_time": 4042.3220067167895,
        "null_rate": 99.98982146949095,
    },
    "other_titles": {
        "avg_changes": 0.0004928479420019201,
        "avg_time": 3045.3938752903773,
        "null_rate": 92.3030644555658,
    },
    "pagination": {
        "avg_changes": 0.012969567159618178,
        "avg_time": 3226.7455410927964,
        "null_rate": 38.3277465112487,
    },
    "physical_dimensions": {
        "avg_changes": 0.00165574165833615,
        "avg_time": 2336.1286827506556,
        "null_rate": 88.82399400935725,
    },
    "physical_format": {
        "avg_changes": 0.0252757864080486,
        "avg_time": 2990.290686930184,
        "null_rate": 74.53942242481871,
    },
    "providers": {
        "avg_changes": 1.0788965164304389e-05,
        "avg_time": 613.073642285292,
        "null_rate": 99.99788212301863,
    },
    "publish_country": {
        "avg_changes": 0.0017686773479458802,
        "avg_time": 5077.199110781045,
        "null_rate": 43.224561148297816,
    },
    "publish_date": {
        "avg_changes": 0.01194275096684623,
        "avg_time": 1554.2227079095205,
        "null_rate": 2.375328294689005,
    },
    "publish_places": {
        "avg_changes": 0.0057898217004331495,
        "avg_time": 1994.341504161254,
        "null_rate": 42.99474479698695,
    },
    "publishers": {
        "avg_changes": 0.02237455638050363,
        "avg_time": 3457.3003991145065,
        "null_rate": 3.8426022289713377,
    },
    "purchase_url": {
        "avg_changes": 4.639866354585907e-06,
        "avg_time": 3981.4979396974913,
        "null_rate": 99.99811781007027,
    },
    "revision": {
        "avg_changes": 1.875948197655085,
        "avg_time": 1147.0281870511515,
        "null_rate": 0.0,
    },
    "scan_on_demand": {
        "avg_changes": 3.4776351902837994e-07,
        "avg_time": 2898.1287552332524,
        "null_rate": 99.92408579528588,
    },
    "scan_records": {
        "avg_changes": 0.00012964799890521,
        "avg_time": 3205.4473507083862,
        "null_rate": 99.98468383572646,
    },
    "series": {
        "avg_changes": 0.00401957515766585,
        "avg_time": 3204.4294852015537,
        "null_rate": 80.22368259607113,
    },
    "source_records": {
        "avg_changes": 0.7901493616121843,
        "avg_time": 2860.2650136421976,
        "null_rate": 40.68450404134878,
    },
    "subject_people": {
        "avg_changes": 3.4776351902837994e-07,
        "avg_time": 662.1152364651041,
        "null_rate": 98.62561397958798,
    },
    "subject_place": {
        "avg_changes": 2.471291105868943e-05,
        "avg_time": 5342.185810704979,
        "null_rate": 89.61097153597943,
    },
    "subject_places": {
        "avg_changes": 8.704289529433181e-07,
        "avg_time": 381.2970664955325,
        "null_rate": 96.05146727593328,
    },
    "subject_time": {
        "avg_changes": 2.263168055640368e-06,
        "avg_time": 4973.765566830855,
        "null_rate": 97.21149746241882,
    },
    "subject_times": {
        "avg_changes": 3.482787349256266e-07,
        "avg_time": 438.968282900625,
        "null_rate": 98.83034203735349,
    },
    "subjects": {
        "avg_changes": 0.03449360606439241,
        "avg_time": 4599.300899736925,
        "null_rate": 37.794880971557845,
    },
    "subtitle": {
        "avg_changes": 0.004560643398696529,
        "avg_time": 2482.717633309411,
        "null_rate": 59.20158807514798,
    },
    "table_of_contents": {
        "avg_changes": 0.032701508030608215,
        "avg_time": 2178.1050423741062,
        "null_rate": 96.16404534155853,
    },
    "title": {
        "avg_changes": 0.0662905959070669,
        "avg_time": 4760.336095255412,
        "null_rate": 0.007921608094920686,
    },
    "title_prefix": {
        "avg_changes": 0.04906178010431137,
        "avg_time": 5086.026896559968,
        "null_rate": 93.72749750541196,
    },
    "translated_from": {
        "avg_changes": 0.0010410571170480598,
        "avg_time": 2004.8195190569927,
        "null_rate": 99.85105315709933,
    },
    "translation_of": {
        "avg_changes": 0.00088546562758017,
        "avg_time": 2024.7585332277517,
        "null_rate": 99.92523656153789,
    },
    "type_key": {"avg_changes": 0.0, "avg_time": None, "null_rate": 0.0},
    "uri_descriptions": {
        "avg_changes": 0.00369119337377338,
        "avg_time": 5404.605885777111,
        "null_rate": 98.712848373139,
    },
    "uris": {
        "avg_changes": 0.00372321614565784,
        "avg_time": 5390.546379631029,
        "null_rate": 98.70567169031901,
    },
    "url": {
        "avg_changes": 8.874087132452107e-06,
        "avg_time": 4857.297609648498,
        "null_rate": 97.73399530375015,
    },
    "volumes": {
        "avg_changes": 5.224789519709643e-06,
        "avg_time": 582.3981145121373,
        "null_rate": 99.99868563770113,
    },
    "weight": {
        "avg_changes": 0.00126623703007436,
        "avg_time": 1264.6373365631193,
        "null_rate": 81.71725514578547,
    },
    "word_count": {
        "avg_changes": 0.0,
        "avg_time": None,
        "null_rate": 99.99871096001995,
    },
    "work_title": {
        "avg_changes": 2.784695053743439e-05,
        "avg_time": 5483.656414422365,
        "null_rate": 98.91260230617489,
    },
    "work_titles": {
        "avg_changes": 4.9250866263229766e-05,
        "avg_time": 4044.6118644717003,
        "null_rate": 98.18311092026931,
    },
    "works": {
        "avg_changes": 0.423037923586201,
        "avg_time": 3865.3224558852153,
        "null_rate": 20.85877016971876,
    },
}

metrics = [timeliness_heinrich.__name__]
metric_configs: List[str | None | MetricConfig] = [
    timeliness_heinrich_config(
        decline_rate_per_column={
            col: stats["avg_changes"] / stats["avg_time"]
            for col, stats in sorted(
                list(ol_column_change_rates.items()),
                key=lambda item: (
                    item[1]["avg_changes"] / item[1]["avg_time"]
                    if item[1]["avg_time"] is not None
                    and item[1]["avg_changes"] is not None
                    and item[0]
                    not in ["last_modified", "latest_revision", "revision", "created"]
                    else 0
                ),
                reverse=True,
            )[:5]
            if stats["avg_time"] is not None and stats["avg_changes"] is not None
        },
        ingestion_date_column="last_modified",
        simulated_assessment_date="2026-01-01",  # newest entry in open library data: 2025-12-31T22:00:00.274823
    ),
]

execute_run(
    folder="/Users/jberndt/Documents/Masterarbeit/Metis/data/local/polluted/20260127_224826/timeliness",
    metrics=metrics,
    metric_configs=metric_configs,
    datasets=["open-library-b0"],
)

/Users/jberndt/Documents/Masterarbeit/Metis/notebooks/../metis/loader/csv_loader.py:16: DtypeWarning: Columns (1,2,3,4,8,9,12,17,20,22,24,26,28,29,30,36,37,39,40,41,42,43,44,45,46,47,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,77,79,80) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(
INFO:metis:Assessing metric 'timeliness_heinrich'
INFO:metis:Writing 2870425 results using CSVWriter
INFO:metis:Results saved to /Users/jberndt/Documents/Masterarbeit/Metis/notebooks/results/20260127_231441/dq_results.csv


DQ run completed in 0:00:45.723989
